# Annexe 10.2 : utilisation d'outils (tool use)

- [Cours](#cours)
- [Exercices](#exercices)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer la variable API_KEY depuis le store IPython
%store -r API_KEY

client = anthropic.Anthropic(api_key=API_KEY)

# Réécrite pour appeler Claude 3 Sonnet, généralement meilleur en tool use, et inclure stop_sequences
def get_completion(messages, system_prompt="", prefill="",stop_sequences=None):
    message = client.messages.create(
        model="claude-3-sonnet-20240229",
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=messages,
        stop_sequences=stop_sequences
    )
    return message.content[0].text

---

## Cours

Même si cela peut paraître complexe au premier abord, l'utilisation d'outils, aussi appelée appel de fonctions (function calling), est en réalité assez simple ! Vous possédez déjà toutes les compétences nécessaires : il s'agit simplement d'une combinaison de substitution et de chaînage de prompts.

Dans les exercices de substitution précédents, nous insérions du texte dans les prompts. Avec l'utilisation d'outils, nous insérons les résultats d'un outil ou d'une fonction. Claude ne peut pas appeler ni accéder lui-même à des outils ou fonctions. À la place, nous lui faisons :
1. produire le nom de l'outil et les arguments qu'il veut utiliser ;
2. interrompre la génération pendant l'appel de l'outil ;
3. puis nous le relançons avec les résultats de l'outil ajoutés au prompt.

> **Note de traduction :** cette annexe présente un ancien format, fondé sur des balises XML dans le prompt système. L'API Claude dispose aujourd'hui d'un support natif du tool use (paramètre `tools`, blocs `tool_use` et `tool_result`), plus fiable. Le principe reste identique. Voir la [documentation tool use](https://docs.claude.com/en/docs/build-with-claude/tool-use).

L'appel de fonctions est utile parce qu'il étend les capacités de Claude et lui permet de traiter des tâches beaucoup plus complexes, en plusieurs étapes.
Quelques exemples de fonctions à fournir à Claude :
- une calculatrice ;
- un compteur de mots ;
- des requêtes et extractions sur une base de données SQL ;
- une API météo.

Pour faire utiliser des outils à Claude, on combine deux éléments :

1. Un prompt système, dans lequel on explique à Claude le principe de l'utilisation d'outils et on lui fournit la liste détaillée des outils à sa disposition.
2. La logique de contrôle qui orchestre et exécute les demandes d'outils de Claude.

### Feuille de route du tool use

*Cette leçon présente notre format actuel d'utilisation d'outils. Nous allons cependant faire évoluer cette fonctionnalité prochainement, notamment avec :*
* *un format plus simple pour définir et appeler les fonctions ;*
* *une gestion plus robuste des erreurs et des cas limites ;*
* *une intégration plus étroite avec le reste de notre API ;*
* *une meilleure fiabilité et de meilleures performances, en particulier pour les tâches complexes.*

### Exemples

Pour activer l'utilisation d'outils, on commence par le prompt système. Dans ce prompt système spécifique, on explique à Claude :
* le principe de l'utilisation d'outils et ce qu'elle implique ;
* comment appeler et utiliser les outils fournis ;
* la liste détaillée des outils disponibles dans ce scénario précis.

Voici la première partie du prompt système, qui explique le tool use à Claude. Elle est réutilisable dans tous les prompts qui font appel à des outils. La structure d'appel que nous donnons à Claude (`<function_calls> [...] </function_calls>`) est une structure qu'il a été spécifiquement entraîné à utiliser : nous vous recommandons de la conserver.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

Voici la seconde partie du prompt système, qui définit précisément les outils à la disposition de Claude dans ce cas. Ici, nous lui donnons une calculatrice qui prend trois paramètres : deux opérandes et un opérateur.

Nous combinons ensuite les deux parties du prompt système.

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

Nous pouvons maintenant poser à Claude une question qui nécessite l'outil `calculator`. Nous plaçons `<function_calls\>` dans `stop_sequences` pour détecter si et quand Claude appelle la fonction.

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Obtenir la réponse de Claude
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

Nous pouvons maintenant extraire les paramètres de l'appel de fonction de Claude et exécuter réellement la fonction à sa place.

Commençons par définir le code de la fonction.

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

Extrayons ensuite les paramètres de la réponse de Claude. Si tous les paramètres sont présents, nous exécutons la calculatrice.

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

Une fois le résultat obtenu, il faut le formater correctement pour que Claude, en le recevant, sache à quel outil il correspond. Il existe un format défini que Claude a appris à reconnaître :
```
<function_results>
<result>
<tool_name>{TOOL_NAME}</tool_name>
<stdout>
{TOOL_RESULT}
</stdout>
</result>
</function_results>
```

Exécutez la cellule ci-dessous pour mettre le résultat de l'outil dans cette structure.

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

Il ne reste plus qu'à renvoyer ce résultat à Claude en l'ajoutant à la même suite de messages que précédemment, et le tour est joué !

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Construire la conversation complète
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Afficher la réponse de Claude
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- FINAL RESULT -------------")
print(final_response)

Bravo, vous avez exécuté une chaîne complète d'utilisation d'outil de bout en bout !

Et si l'on pose à Claude une question qui ne nécessite pas du tout l'outil fourni ?

In [ ]:
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Obtenir la réponse de Claude
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

Réussi ! Comme vous le voyez, Claude a compris qu'il ne fallait pas appeler la fonction quand ce n'était pas nécessaire.

Si vous voulez expérimenter avec les prompts du cours sans modifier le contenu ci-dessus, descendez tout en bas du notebook jusqu'à l'[**Espace de test**](#espace-de-test).

---

## Exercices
- [Exercice 10.2.1 - SQL](#exercice-1021---sql)

### Exercice 10.2.1 - SQL
Dans cet exercice, vous rédigez un prompt d'utilisation d'outils pour interroger et alimenter la plus petite « base de données » du monde. Voici la base initialisée, qui n'est en réalité qu'un dictionnaire.

In [ ]:
db = {
    "users": [
        {"id": 1, "name": "Alice", "email": "alice@example.com"},
        {"id": 2, "name": "Bob", "email": "bob@example.com"},
        {"id": 3, "name": "Charlie", "email": "charlie@example.com"}
    ],
    "products": [
        {"id": 1, "name": "Widget", "price": 9.99},
        {"id": 2, "name": "Gadget", "price": 14.99},
        {"id": 3, "name": "Doohickey", "price": 19.99}
    ]
}

Et voici le code des fonctions qui lisent et écrivent dans la base.

In [ ]:
def get_user(user_id):
    for user in db["users"]:
        if user["id"] == user_id:
            return user
    return None

def get_product(product_id):
    for product in db["products"]:
        if product["id"] == product_id:
            return product
    return None

def add_user(name, email):
    user_id = len(db["users"]) + 1
    user = {"id": user_id, "name": name, "email": email}
    db["users"].append(user)
    return user

def add_product(name, price):
    product_id = len(db["products"]) + 1
    product = {"id": product_id, "name": name, "price": price}
    db["products"].append(product)
    return product

Pour résoudre l'exercice, commencez par définir un prompt système sur le modèle de `system_prompt_tools_specific_tools` ci-dessus. Incluez le nom et la description de chaque outil, ainsi que le nom, le type et la description de chaque paramètre de chaque fonction. Un squelette de départ est fourni ci-dessous.

In [ ]:
system_prompt_tools_specific_tools_sql = """
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools_sql

Quand vous êtes prêt, testez votre prompt système de définition d'outils sur les exemples ci-dessous. Exécutez simplement la cellule suivante.

In [ ]:
examples = [
    "Add a user to the database named Deborah.",
    "Add a product to the database named Thingo",
    "Tell me the name of User 2",
    "Tell me the name of Product 3"
]

for example in examples:
    message = {
        "role": "user",
        "content": example
    }

    # Obtenir et afficher la réponse de Claude
    function_calling_response = get_completion([message], system_prompt=system_prompt, stop_sequences=stop_sequences)
    print(example, "\n----------\n\n", function_calling_response, "\n*********\n*********\n*********\n\n")

Si votre solution est correcte, les messages d'appel de fonction doivent appeler correctement les fonctions `add_user`, `add_product`, `get_user` et `get_product`.

Pour aller plus loin, ajoutez des cellules de code et écrivez le code d'analyse des paramètres. Appelez ensuite les fonctions avec les paramètres fournis par Claude pour voir l'état de la « base de données » après chaque appel.

❓ Pour voir une solution possible, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_10_2_1_solution; print(exercise_10_2_1_solution)

### Bravo !

Vous maîtrisez maintenant l'utilisation d'outils et l'appel de fonctions ! Passez à la dernière annexe pour en savoir plus sur la recherche et le RAG.

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Obtenir la réponse de Claude
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Construire la conversation complète
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Afficher la réponse de Claude
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- FINAL RESULT -------------")
print(final_response)

In [ ]:
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Obtenir la réponse de Claude
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)